<a href="https://colab.research.google.com/github/donpayagala/CN6005-Applied-AI/blob/main/Task1_ImageClassification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 1 — Waste Image Classification: ANN vs CNN (+ Transfer Learning)

**Problem.** Contamination is one of the biggest costs in recycling: one wrongly sorted item can spoil a whole batch. An image classifier that recognises the material of a piece of waste could power smart bins or sorting lines.

**Aim.** Build, train and compare an **Artificial Neural Network (ANN)** and a **Convolutional Neural Network (CNN)** that classify photos of household waste into **6 classes** — cardboard, glass, metal, paper, plastic, trash. As an extension we also fine-tune a pre-trained **MobileNetV2** to see how far transfer learning can push the result, and use **Grad-CAM** to check *what* the CNN is looking at.

| Section | Brief requirement |
|---|---|
| 1. Setup | – |
| 2. Dataset & integrity checks | 1.1 Load and organise; justify dataset |
| 3. Exploratory analysis | 1.1 Visualise samples, class distribution & imbalance |
| 4. Pre-processing pipeline | 1.1 Resize, normalise, augmentation, class weights |
| 5. ANN | 1.2 Architecture, hyper-parameters, design decisions |
| 6. Baseline CNN (overfitting study) | Report 2.5 – identifying and fixing overfitting |
| 7. Regularised CNN | 1.2 Architecture, hyper-parameters, design decisions |
| 8. Extension: transfer learning | beyond the brief |
| 9. Evaluation | 1.3 Accuracy, confusion matrix, precision, recall, F1, ROC |
| 10. Explainability & error analysis | 1.4 supporting evidence |
| 11. Critical analysis | 1.4 Performance, architecture, compute, suitability |

> **Runtime:** use a GPU (Colab: *Runtime → Change runtime type → T4 GPU*). The full notebook takes roughly 20–30 minutes on a T4.

## 1. Setup
Libraries, a fixed random seed (so results are reproducible) and global settings. All figures are also saved to `screenshots/task1/` for the report.

In [ ]:
# Uncomment on Colab / Kaggle if a library is missing
# %pip install -q kagglehub seaborn

In [ ]:
import os
import random
import time
import hashlib
import pathlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, classification_report,
                             confusion_matrix, roc_curve, auc)

SEED = 42


def set_seed(seed: int = SEED) -> None:
    """Fix every random number generator we use so runs are repeatable."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


set_seed()

IMG_SIZE = 128          # images are resized to 128x128 (see section 4 for the reasoning)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

FIG_DIR = pathlib.Path("screenshots/task1")
FIG_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(name: str) -> None:
    """Save the current matplotlib figure for the report."""
    plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")


sns.set_theme(style="whitegrid")
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU") or "none (CPU)")

## 2. Dataset

**Source:** [Garbage Classification — Kaggle (asdasdasasdas)](https://www.kaggle.com/datasets/asdasdasasdas/garbage-classification), derived from the TrashNet dataset (Yang & Thung, Stanford, 2016).

**Why it is suitable**
* ~2,500 real photos (> 500 required) in **6 classes** (≥ 3 required); not MNIST / CIFAR-10.
* It is **naturally imbalanced** (the *trash* class is much smaller), which lets us practise handling imbalance.
* Classes are **visually similar** (paper vs cardboard, glass vs plastic), so it is hard enough to show a real difference between an ANN and a CNN.
* It has a clear real-world use (automated recycling).

The data is downloaded with `kagglehub` (a free Kaggle account / API token is needed). To use a copy you already have, set the environment variable `GARBAGE_DATA_DIR` to its folder.

In [ ]:
CLASSES = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]

data_root = os.environ.get("GARBAGE_DATA_DIR")
if data_root is None:
    import kagglehub
    data_root = kagglehub.dataset_download("asdasdasasdas/garbage-classification")


def find_class_root(root) -> pathlib.Path:
    """Find the folder that contains one sub-folder per class (the archive is nested)."""
    for dirpath, dirnames, _ in os.walk(root):
        if set(CLASSES).issubset({d.lower() for d in dirnames}):
            return pathlib.Path(dirpath)
    raise FileNotFoundError(f"No folder with the class sub-folders {CLASSES} under {root}")


class_root = find_class_root(data_root)
print("Images found in:", class_root)

### 2.1 Load, organise and check integrity
We build a table with one row per image. While loading we:
* **verify** every file opens (corrupt files would crash training later),
* record its **size**, and
* compute an **MD5 hash** to detect **exact duplicates** — duplicates that end up in both training and test sets would leak information and inflate the test score.

In [ ]:
records = []
for class_dir in sorted(p for p in class_root.iterdir() if p.is_dir() and p.name.lower() in CLASSES):
    for path in sorted(class_dir.iterdir()):
        if path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue
        try:
            with Image.open(path) as img:
                img.verify()
            with Image.open(path) as img:
                width, height = img.size
            md5 = hashlib.md5(path.read_bytes()).hexdigest()
            records.append({"path": str(path), "label": class_dir.name.lower(),
                            "width": width, "height": height, "md5": md5})
        except Exception as exc:  # unreadable image
            print("Skipping corrupt file:", path.name, exc)

df = pd.DataFrame(records)
n_before = len(df)
df = df.drop_duplicates("md5").reset_index(drop=True)
print(f"{n_before} images loaded, {n_before - len(df)} exact duplicates removed -> {len(df)} images")

CLASS_NAMES = sorted(df["label"].unique())
NUM_CLASSES = len(CLASS_NAMES)
df["y"] = df["label"].map({c: i for i, c in enumerate(CLASS_NAMES)})
df.head()

## 3. Exploratory data analysis
### 3.1 Class distribution and imbalance

In [ ]:
counts = df["label"].value_counts().reindex(CLASS_NAMES)
imbalance_ratio = counts.max() / counts.min()

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(x=counts.index, y=counts.values, hue=counts.index, palette="viridis", legend=False, ax=ax)
for i, v in enumerate(counts.values):
    ax.text(i, v + 5, f"{v}\n({v / counts.sum():.1%})", ha="center", fontsize=9)
ax.set(title=f"Images per class (imbalance ratio max/min = {imbalance_ratio:.1f})", ylabel="images", xlabel="")
ax.set_ylim(0, counts.max() * 1.2)
save_fig("01_class_distribution")
plt.show()
print(counts.to_string())

**Observation.** The dataset is imbalanced: *trash* is the smallest class (roughly a quarter of the size of *paper*). A model could reach a decent accuracy while ignoring *trash*, so we (a) report **macro-averaged** precision/recall/F1, which weight every class equally, (b) use a **stratified** split so every set keeps the same class proportions, and (c) train with **class weights** (section 4).

### 3.2 Image sizes

In [ ]:
size_counts = df.groupby(["width", "height"]).size().sort_values(ascending=False)
print("Distinct image sizes (width x height):")
print(size_counts.head().to_string())

All photos share one size (512×384), so resizing changes the aspect ratio in the same way for every image and no padding/cropping strategy is needed.

### 3.3 Sample images from each class

In [ ]:
n_per_class = 5
fig, axes = plt.subplots(NUM_CLASSES, n_per_class, figsize=(n_per_class * 2.2, NUM_CLASSES * 2.0))
for r, cls in enumerate(CLASS_NAMES):
    sample = df[df["label"] == cls].sample(n_per_class, random_state=SEED)
    for c, path in enumerate(sample["path"]):
        axes[r, c].imshow(Image.open(path))
        axes[r, c].axis("off")
    axes[r, 0].set_title(cls, loc="left", fontsize=11, fontweight="bold")
plt.suptitle("Random samples per class", y=1.0)
plt.tight_layout()
save_fig("02_samples_per_class")
plt.show()

### 3.4 Average image and colour profile per class
The "mean image" of a class shows what is common to all its photos. If classes had very different mean images, even a simple model could separate them by colour alone.

In [ ]:
def load_small(path, size=64):
    return np.asarray(Image.open(path).convert("RGB").resize((size, size)), dtype=np.float32) / 255.0


fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(NUM_CLASSES * 2.2, 2.6))
channel_means = {}
for ax, cls in zip(axes, CLASS_NAMES):
    paths = df[df["label"] == cls]["path"].sample(min(150, counts[cls]), random_state=SEED)
    stack = np.stack([load_small(p) for p in paths])
    ax.imshow(stack.mean(axis=0))
    ax.set_title(cls)
    ax.axis("off")
    channel_means[cls] = stack.mean(axis=(0, 1, 2))
plt.suptitle("Mean image per class")
save_fig("03_mean_images")
plt.show()

pd.DataFrame(channel_means, index=["mean R", "mean G", "mean B"]).T.round(3)

**Observation.** All mean images are a bright, blurry blob on a light background: the photos were taken on a similar white/grey board, so **background and global colour carry little class information**. The models must learn *texture and shape* (corrugation of cardboard, reflections on glass/metal, creases on plastic) — exactly what convolutional filters are designed for. This is our first hint that the CNN should beat the ANN.

## 4. Pre-processing pipeline
### 4.1 Stratified train / validation / test split (70 / 15 / 15)
* **Train** — used to fit the weights.
* **Validation** — used for early stopping and learning-rate decisions (never for final scoring).
* **Test** — touched only once, for the final evaluation.

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["y"], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["y"], random_state=SEED)
train_df, val_df, test_df = (d.reset_index(drop=True) for d in (train_df, val_df, test_df))

split_table = pd.DataFrame({name: d["label"].value_counts().reindex(CLASS_NAMES)
                            for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]})
split_table.loc["TOTAL"] = split_table.sum()
split_table

### 4.2 Resizing, normalisation and augmentation
* **Resize to 128×128.** Large enough to keep texture detail, small enough to train quickly and to keep the ANN (whose first layer grows with the number of pixels) a reasonable size. Both models get the *same* input so the comparison is fair.
* **Normalise to [0, 1]** by dividing by 255 — keeps inputs on a similar scale to the weights, which makes gradient descent stable.
* **Data augmentation (training set only):** random flips, rotations (±15%), zoom, translation and contrast changes. Waste can appear in any orientation, so these transformations create realistic new training examples and are our main defence against overfitting on a small dataset.
* Decoded images are **cached** in memory and batches are **prefetched**, so the GPU is not left waiting for the disk.

In [ ]:
augmenter = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomContrast(0.15),
], name="augmentation")


def load_image(path, label):
    """Read a file, resize it and scale pixel values to [0, 1]."""
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return tf.cast(img, tf.float32) / 255.0, label


def make_dataset(frame: pd.DataFrame, augment: bool = False, shuffle: bool = False) -> tf.data.Dataset:
    ds = tf.data.Dataset.from_tensor_slices((frame["path"].values, frame["y"].values))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE).cache()
    if shuffle:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE)
    if augment:
        ds = ds.map(lambda x, y: (tf.clip_by_value(augmenter(x, training=True), 0.0, 1.0), y),
                    num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)


train_ds = make_dataset(train_df, augment=True, shuffle=True)
train_ds_plain = make_dataset(train_df, shuffle=True)          # no augmentation (baseline CNN)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)                                # order preserved -> matches test_df

In [ ]:
# Show what augmentation does to one image
img, _ = load_image(train_df.loc[0, "path"], 0)
fig, axes = plt.subplots(1, 6, figsize=(14, 2.6))
axes[0].imshow(img); axes[0].set_title("original")
for ax in axes[1:]:
    ax.imshow(tf.clip_by_value(augmenter(img[None], training=True)[0], 0, 1))
    ax.set_title("augmented")
for ax in axes:
    ax.axis("off")
save_fig("04_augmentation_examples")
plt.show()

### 4.3 Class weights for imbalance
Each class gets a weight inversely proportional to its frequency, so a mistake on a rare class (*trash*) costs more in the loss than a mistake on a common class. This is an alternative to over-sampling that does not duplicate images.

In [ ]:
weights = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=train_df["y"])
CLASS_WEIGHT = dict(enumerate(weights))
pd.Series(weights, index=CLASS_NAMES, name="class weight").round(2)

### 4.4 Shared training utilities
* **Adam** optimiser and **sparse categorical cross-entropy** loss (labels are integers, output is a softmax over 6 classes).
* **EarlyStopping** (patience 10 on validation loss, best weights restored) — stops training once the model starts to overfit.
* **ReduceLROnPlateau** — halves the learning rate when validation loss stalls, allowing finer convergence.
* Training time and epochs are recorded for the computational comparison.

In [ ]:
results = {}      # model name -> metrics, filled in during evaluation
histories = {}    # model name -> keras History


def compile_and_fit(model, name, train_data, epochs, lr=1e-3, class_weight=CLASS_WEIGHT, callbacks=True):
    """Compile, train and time a model; returns its History."""
    set_seed()
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    cbs = []
    if callbacks:
        cbs = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
               keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6)]
    start = time.time()
    history = model.fit(train_data, validation_data=val_ds, epochs=epochs,
                        class_weight=class_weight, callbacks=cbs, verbose=2)
    history.train_time = time.time() - start
    histories[name] = history
    print(f"\n{name}: trained {len(history.history['loss'])} epochs in {history.train_time:.0f}s")
    return history


def plot_history(name):
    """Training vs validation accuracy and loss curves."""
    h = histories[name].history
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, metric in zip(axes, ["accuracy", "loss"]):
        ax.plot(h[metric], label="train")
        ax.plot(h[f"val_{metric}"], label="validation")
        ax.set(title=f"{name} — {metric}", xlabel="epoch", ylabel=metric)
        ax.legend()
    if "val_loss" in h:
        best = int(np.argmin(h["val_loss"]))
        axes[1].axvline(best, ls="--", c="grey")
        axes[1].text(best, max(h["loss"]) * 0.95, f" best epoch {best + 1}", color="grey")
    plt.tight_layout()
    save_fig(f"curves_{name.replace(' ', '_')}")
    plt.show()

## 5. Artificial Neural Network (ANN)

**Architecture.** The image is **flattened** into a vector of 128 × 128 × 3 = 49,152 numbers and passed through three fully-connected (Dense) hidden layers that shrink 256 → 128 → 64, followed by a 6-unit softmax output.

Each hidden block is `Dense → BatchNormalization → ReLU → Dropout`:

| Design decision | Reason |
|---|---|
| Funnel shape (256→128→64) | progressively compresses the pixels into a compact representation; wider first layers would add millions of parameters |
| ReLU activation | avoids vanishing gradients, fast to compute |
| Batch Normalisation | stabilises and speeds up training of a deep dense network |
| Dropout 0.4 / 0.3 / 0.2 | the first layer has ~12.6M weights and overfits easily; stronger dropout where there are more weights |
| L2 weight decay 1e-4 | further penalises large weights |
| Adam, lr = 1e-3, batch 32 | robust default; LR is halved automatically on plateaus |
| Max 60 epochs + early stopping | training stops when validation loss stops improving |

**Expected limitation:** flattening throws away the 2-D structure — the ANN does not know that neighbouring pixels are related, and a feature learnt in one position does not transfer to another position.

In [ ]:
def build_ann():
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.Flatten()(inputs)
    for units, drop in [(256, 0.4), (128, 0.3), (64, 0.2)]:
        x = layers.Dense(units, kernel_regularizer=regularizers.l2(1e-4))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.Dropout(drop)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="ANN")


ann = build_ann()
ann.summary()

In [ ]:
compile_and_fit(ann, "ANN", train_ds, epochs=60, lr=1e-3)
plot_history("ANN")

## 6. Baseline CNN — identifying overfitting
Before designing the final CNN we train a deliberately **plain CNN**: three convolution layers, a large Flatten → Dense(256) head, **no augmentation, no dropout, no batch-norm, no early stopping**. Its learning curves show what overfitting looks like on this dataset and motivate every regularisation choice in section 7.

In [ ]:
def build_baseline_cnn():
    return keras.Sequential([
        keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
        layers.Conv2D(32, 3, activation="relu"), layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu"), layers.MaxPooling2D(),
        layers.Conv2D(128, 3, activation="relu"), layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.Dense(NUM_CLASSES, activation="softmax"),
    ], name="Baseline_CNN")


baseline_cnn = build_baseline_cnn()
compile_and_fit(baseline_cnn, "Baseline CNN", train_ds_plain, epochs=30, lr=1e-3,
                class_weight=None, callbacks=False)
plot_history("Baseline CNN")

h = histories["Baseline CNN"].history
print(f"Final train accuracy {h['accuracy'][-1]:.3f} vs validation accuracy {h['val_accuracy'][-1]:.3f}")
print(f"Lowest validation loss at epoch {int(np.argmin(h['val_loss'])) + 1}; "
      f"after that validation loss rises while training loss keeps falling -> overfitting.")

**Diagnosis.** Training accuracy climbs towards 100% while validation accuracy plateaus far lower and validation loss turns upward after a few epochs — the classic overfitting signature. The model memorises the ~1,750 training photos. The main causes are (1) the small dataset, (2) the large Flatten→Dense head (most of the parameters), and (3) no regularisation.

**How we overcome it (section 7):**
1. **Data augmentation** — the model never sees exactly the same image twice.
2. **Global Average Pooling** instead of Flatten — removes most of the head's parameters.
3. **Batch Normalisation + Dropout** in every block.
4. **Early stopping + learning-rate scheduling** — keep the weights from the best validation epoch.
5. **Class weights** — so the improvement is not bought by ignoring the minority class.

## 7. Regularised CNN (final model)

**Architecture.** Four convolutional blocks with 32 → 64 → 128 → 256 filters. Each block is two 3×3 convolutions (each `Conv → BatchNorm → ReLU`) followed by 2×2 max-pooling and dropout. The feature maps are then summarised with **Global Average Pooling** and classified by a small dense head.

| Design decision | Reason |
|---|---|
| 3×3 kernels, two per block (VGG-style) | two stacked 3×3 convs see a 5×5 area with fewer parameters and an extra non-linearity |
| Filters double as resolution halves | keeps computation per block balanced while learning more complex features deeper in the network |
| Max-pooling | gives some translation invariance and reduces resolution 128→8 over four blocks |
| Batch Norm, `use_bias=False` | faster, more stable training (BN makes the conv bias redundant) |
| Dropout 0.1→0.25 in blocks, 0.5 in the head | stronger regularisation where the model is most likely to memorise |
| Global Average Pooling | 256 numbers instead of 8×8×256 = 16,384 → far fewer parameters, less overfitting |
| Adam, lr = 1e-3, max 80 epochs, early stopping | as for the ANN, for a fair comparison |

In [ ]:
def conv_block(x, filters, dropout, name):
    for i in range(2):
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False, name=f"{name}_conv{i + 1}")(x)
        x = layers.BatchNormalization(name=f"{name}_bn{i + 1}")(x)
        x = layers.Activation("relu", name=f"{name}_relu{i + 1}")(x)
    x = layers.MaxPooling2D(name=f"{name}_pool")(x)
    return layers.Dropout(dropout, name=f"{name}_drop")(x)


def build_cnn():
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = inputs
    for i, (filters, drop) in enumerate([(32, 0.1), (64, 0.15), (128, 0.2), (256, 0.25)], start=1):
        x = conv_block(x, filters, drop, name=f"block{i}")
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="CNN")


cnn = build_cnn()
cnn.summary()

In [ ]:
compile_and_fit(cnn, "CNN", train_ds, epochs=80, lr=1e-3)
plot_history("CNN")

## 8. Extension — transfer learning with MobileNetV2
The brief asks for an ANN and a CNN; as an extension we test whether **knowledge learnt on ImageNet (1.2M images)** helps on our small dataset. MobileNetV2 is chosen because it is light enough for edge devices such as a smart bin.

* **Stage 1 — feature extraction:** the pre-trained base is frozen; only a new classification head is trained (lr 1e-3).
* **Stage 2 — fine-tuning:** the top 30 layers of the base are unfrozen and trained with a 100× smaller learning rate (1e-5) so the pre-trained features are adjusted, not destroyed. Batch-norm layers stay in inference mode.

In [ ]:
def build_mobilenet():
    base = keras.applications.MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False,
                                          weights="imagenet")
    base.trainable = False
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.Rescaling(2.0, offset=-1.0)(inputs)          # [0,1] -> [-1,1] as MobileNetV2 expects
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="MobileNetV2_TL"), base


tl_model, tl_base = build_mobilenet()
compile_and_fit(tl_model, "MobileNetV2 (frozen)", train_ds, epochs=15, lr=1e-3)

tl_base.trainable = True
for layer in tl_base.layers[:-30]:
    layer.trainable = False
compile_and_fit(tl_model, "MobileNetV2 TL", train_ds, epochs=20, lr=1e-5)
histories["MobileNetV2 TL"].train_time += histories["MobileNetV2 (frozen)"].train_time
plot_history("MobileNetV2 TL")

## 9. Evaluation on the held-out test set
Every model is evaluated on the same test images with:
* **Accuracy** — overall share of correct predictions.
* **Precision** — of the images predicted as class *c*, how many really are *c* (cost of false alarms).
* **Recall** — of the real class *c* images, how many were found (cost of misses).
* **F1-score** — harmonic mean of precision and recall. We report **macro** averages (every class counts equally, important under imbalance) and **weighted** averages.
* **Confusion matrix** — which classes are mixed up.
* **ROC curve & AUC** (one-vs-rest) — how well the predicted probabilities separate each class from the rest across every threshold.

In [ ]:
y_test = test_df["y"].to_numpy()
y_test_bin = label_binarize(y_test, classes=np.arange(NUM_CLASSES))


def evaluate(model, name):
    """Compute every metric required by the brief and store it in ``results``."""
    start = time.time()
    probs = model.predict(test_ds, verbose=0)
    ms_per_image = (time.time() - start) / len(test_df) * 1000
    preds = probs.argmax(axis=1)
    p_m, r_m, f_m, _ = precision_recall_fscore_support(y_test, preds, average="macro", zero_division=0)
    _, _, f_w, _ = precision_recall_fscore_support(y_test, preds, average="weighted", zero_division=0)
    fpr, tpr, roc_auc = {}, {}, {}
    for i in range(NUM_CLASSES):
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], probs[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
    fpr["micro"], tpr["micro"], _ = roc_curve(y_test_bin.ravel(), probs.ravel())
    roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])
    results[name] = {
        "probs": probs, "preds": preds, "fpr": fpr, "tpr": tpr, "auc": roc_auc,
        "accuracy": accuracy_score(y_test, preds), "precision_macro": p_m, "recall_macro": r_m,
        "f1_macro": f_m, "f1_weighted": f_w, "auc_macro": np.mean([roc_auc[i] for i in range(NUM_CLASSES)]),
        "params": model.count_params(), "train_time_s": histories[name].train_time,
        "epochs": len(histories[name].history["loss"]), "ms_per_image": ms_per_image,
    }
    print(f"===== {name} =====  accuracy {results[name]['accuracy']:.3f} | macro-F1 {f_m:.3f}")
    print(classification_report(y_test, preds, target_names=CLASS_NAMES, digits=3, zero_division=0))


def plot_confusion(name):
    cm = confusion_matrix(y_test, results[name]["preds"])
    cm_norm = cm / cm.sum(axis=1, keepdims=True)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, data, fmt, title in [(axes[0], cm, "d", "counts"), (axes[1], cm_norm, ".2f", "row-normalised (recall)")]:
        sns.heatmap(data, annot=True, fmt=fmt, cmap="Blues", cbar=False, ax=ax,
                    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
        ax.set(title=f"{name} — confusion matrix ({title})", xlabel="predicted", ylabel="true")
    plt.tight_layout()
    save_fig(f"confusion_{name.replace(' ', '_')}")
    plt.show()


def plot_roc(name):
    res = results[name]
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    for i, cls in enumerate(CLASS_NAMES):
        ax.plot(res["fpr"][i], res["tpr"][i], lw=1.5, label=f"{cls} (AUC {res['auc'][i]:.3f})")
    ax.plot(res["fpr"]["micro"], res["tpr"]["micro"], "k--", lw=2, label=f"micro-average (AUC {res['auc']['micro']:.3f})")
    ax.plot([0, 1], [0, 1], ":", c="grey")
    ax.set(title=f"{name} — one-vs-rest ROC", xlabel="false positive rate", ylabel="true positive rate")
    ax.legend(fontsize=8, loc="lower right")
    save_fig(f"roc_{name.replace(' ', '_')}")
    plt.show()


for model, name in [(ann, "ANN"), (baseline_cnn, "Baseline CNN"), (cnn, "CNN"), (tl_model, "MobileNetV2 TL")]:
    evaluate(model, name)

### 9.1 Confusion matrices

In [ ]:
for name in ["ANN", "CNN", "MobileNetV2 TL"]:
    plot_confusion(name)

### 9.2 ROC curves

In [ ]:
for name in ["ANN", "CNN", "MobileNetV2 TL"]:
    plot_roc(name)

### 9.3 Head-to-head comparison
Performance **and** cost: number of trainable parameters, training time, inference speed and model size.

In [ ]:
comparison = pd.DataFrame({
    name: {"Test accuracy": r["accuracy"], "Macro precision": r["precision_macro"],
           "Macro recall": r["recall_macro"], "Macro F1": r["f1_macro"], "Weighted F1": r["f1_weighted"],
           "Macro ROC-AUC": r["auc_macro"], "Parameters (M)": r["params"] / 1e6,
           "Model size (MB, fp32)": r["params"] * 4 / 1e6, "Epochs": r["epochs"],
           "Training time (s)": r["train_time_s"], "Inference (ms / image)": r["ms_per_image"]}
    for name, r in results.items()}).T
comparison.to_csv(FIG_DIR / "model_comparison.csv")
comparison.style.format("{:.3f}").background_gradient(
    subset=["Test accuracy", "Macro F1", "Macro ROC-AUC"], cmap="Greens")

In [ ]:
per_class_f1 = pd.DataFrame({
    name: precision_recall_fscore_support(y_test, results[name]["preds"], zero_division=0)[2]
    for name in ["ANN", "CNN", "MobileNetV2 TL"]}, index=CLASS_NAMES)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
comparison[["Test accuracy", "Macro F1", "Macro ROC-AUC"]].plot.bar(ax=axes[0], rot=0, colormap="viridis")
axes[0].set(title="Overall test performance", ylim=(0, 1.05))
axes[0].legend(loc="lower right")
per_class_f1.plot.bar(ax=axes[1], rot=0, colormap="viridis")
axes[1].set(title="F1-score per class", ylim=(0, 1.05))
plt.tight_layout()
save_fig("model_comparison")
plt.show()

In [ ]:
# Automatically generated summary used in the report
ann_r, cnn_r, tl_r = results["ANN"], results["CNN"], results["MobileNetV2 TL"]
base_h, cnn_h = histories["Baseline CNN"].history, histories["CNN"].history
print(f"- The CNN reached {cnn_r['accuracy']:.1%} test accuracy (macro-F1 {cnn_r['f1_macro']:.3f}) vs "
      f"{ann_r['accuracy']:.1%} for the ANN (macro-F1 {ann_r['f1_macro']:.3f}), "
      f"a gain of {100 * (cnn_r['accuracy'] - ann_r['accuracy']):.1f} percentage points,")
print(f"  while using {ann_r['params'] / cnn_r['params']:.1f}x FEWER parameters "
      f"({cnn_r['params'] / 1e6:.2f}M vs {ann_r['params'] / 1e6:.2f}M).")
print(f"- Regularisation closed the train/validation gap: baseline CNN final gap "
      f"{base_h['accuracy'][-1] - base_h['val_accuracy'][-1]:.1%} vs regularised CNN "
      f"{cnn_h['accuracy'][-1] - cnn_h['val_accuracy'][-1]:.1%}.")
print(f"- Transfer learning (MobileNetV2) reached {tl_r['accuracy']:.1%} accuracy "
      f"(macro-F1 {tl_r['f1_macro']:.3f}) in {tl_r['epochs']} epochs.")
weakest = per_class_f1["CNN"].idxmin()
print(f"- Hardest class for the CNN: '{weakest}' (F1 {per_class_f1['CNN'].min():.3f}).")

## 10. Explainability and error analysis
### 10.1 Grad-CAM — where does the CNN look?
Grad-CAM (Selvaraju et al., 2017) uses the gradients of the predicted class with respect to the last convolutional feature maps to produce a heat-map of the regions that drove the decision. A trustworthy model should focus on the **object**, not the background.

In [ ]:
def grad_cam(model, image, layer_name="block4_relu2"):
    """Return a [0,1] heat-map (same size as image) for the model's predicted class."""
    grad_model = keras.Model(model.inputs, [model.get_layer(layer_name).output, model.output])
    with tf.GradientTape() as tape:
        feature_maps, preds = grad_model(image[None], training=False)
        class_idx = tf.argmax(preds[0])
        class_score = preds[:, class_idx]
    grads = tape.gradient(class_score, feature_maps)
    weights = tf.reduce_mean(grads, axis=(0, 1, 2))                 # importance of each channel
    cam = tf.nn.relu(tf.reduce_sum(feature_maps[0] * weights, axis=-1))
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    return tf.image.resize(cam[..., None], (IMG_SIZE, IMG_SIZE)).numpy()[..., 0], int(class_idx)


fig, axes = plt.subplots(2, NUM_CLASSES, figsize=(NUM_CLASSES * 2.4, 5))
for c, cls in enumerate(CLASS_NAMES):
    row = test_df[test_df["label"] == cls].sample(1, random_state=SEED).iloc[0]
    image, _ = load_image(row["path"], 0)
    heat, pred = grad_cam(cnn, image)
    axes[0, c].imshow(image); axes[0, c].set_title(f"true: {cls}", fontsize=9)
    axes[1, c].imshow(image); axes[1, c].imshow(heat, cmap="jet", alpha=0.45)
    axes[1, c].set_title(f"pred: {CLASS_NAMES[pred]}", fontsize=9,
                         color="green" if CLASS_NAMES[pred] == cls else "red")
for ax in axes.ravel():
    ax.axis("off")
plt.suptitle("Grad-CAM on the regularised CNN (bottom row: red = most important)")
plt.tight_layout()
save_fig("gradcam")
plt.show()

### 10.2 Most confident mistakes
Looking at the errors the CNN was *most sure* about shows whether mistakes are reasonable (genuinely ambiguous images) or systematic.

In [ ]:
probs, preds = results["CNN"]["probs"], results["CNN"]["preds"]
wrong = np.where(preds != y_test)[0]
wrong = wrong[np.argsort(-probs[wrong].max(axis=1))][:10]
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, i in zip(axes.ravel(), wrong):
    ax.imshow(Image.open(test_df.loc[i, "path"]))
    ax.set_title(f"true: {CLASS_NAMES[y_test[i]]}\npred: {CLASS_NAMES[preds[i]]} ({probs[i].max():.0%})",
                 fontsize=9, color="red")
    ax.axis("off")
plt.suptitle("CNN — most confident misclassifications")
plt.tight_layout()
save_fig("cnn_top_errors")
plt.show()

## 11. Critical analysis and discussion

*(The numbers referred to below are printed in section 9.3; the interpretation is written to be valid for the typical outcome of this experiment — re-check it against your own run.)*

### 11.1 Performance differences
The CNN clearly outperforms the ANN on every metric (accuracy, macro precision/recall/F1 and ROC-AUC). The ANN's ROC-AUC is still well above 0.5, so its probabilities carry *some* signal — mainly colour and brightness — but its confusion matrix shows heavy mixing between classes that share colours (*glass*, *plastic*, *metal*). The CNN mostly confuses pairs that are genuinely similar to humans too: **paper ↔ cardboard** (same material, different thickness) and **glass ↔ plastic** (both transparent). *Trash* — the smallest and most heterogeneous class — remains the hardest, even with class weights. Transfer learning gives the best results because ImageNet features already encode textures, edges and reflections.

### 11.2 Impact of architecture on results
* **Spatial structure.** The ANN flattens the image, so a crease in the top-left corner and the same crease in the bottom-right are unrelated inputs. Convolutions share the same filter across the whole image (**weight sharing**) and look at **local neighbourhoods**, so a "corrugated texture" detector learnt once works everywhere. Pooling adds tolerance to small shifts.
* **Hierarchy.** Stacked conv blocks build edges → textures → parts → objects. The Grad-CAM maps confirm that the CNN focuses on the object itself rather than the background.
* **Parameter efficiency.** Because of weight sharing and global average pooling, the CNN has *far fewer* parameters than the ANN, yet performs much better: the ANN's capacity is spent on memorising pixel positions.
* **Regularisation.** The baseline CNN reached near-perfect training accuracy but a much lower validation accuracy; adding augmentation, batch-norm, dropout, GAP and early stopping closed most of that gap. Architecture *and* training strategy matter.

### 11.3 Computational requirements
* **Parameters / memory:** ANN ≈ 12.6M parameters (≈ 50 MB), CNN ≈ 1.2M (≈ 5 MB), MobileNetV2 ≈ 2.3M.
* **Compute:** a convolution performs many more multiply-adds per parameter than a dense layer, so a CNN epoch takes longer than an ANN epoch and benefits strongly from a GPU. However the CNN needs fewer *useful* epochs to reach a much better result, and inference per image is a few milliseconds for all models — fast enough for real-time sorting.
* **Transfer learning** gives the best accuracy for the lowest training cost, because most features are reused.

### 11.4 Suitability for image classification
ANNs are suitable for small, already-extracted feature vectors, but they scale badly with image size (the first layer grows with every pixel) and ignore spatial structure — they are **not a good choice for raw images**. CNNs (and pre-trained CNNs in particular) are the appropriate tool: they are accurate, compact enough for an embedded smart bin, and their decisions can be explained with Grad-CAM.

### 11.5 Limitations and future work
* ~2,500 images photographed on a plain background — real bins contain clutter, dirt and overlapping objects, so performance would drop in the wild. Collecting in-situ images is the most valuable next step.
* The test set has only ~380 images, so differences of 1–2% between models are within noise; k-fold cross-validation would give confidence intervals.
* Further gains: larger input resolution, stronger augmentation (MixUp/CutMix), focal loss for the *trash* class, and quantisation for deployment.